# Waste AI v6 — 8 classes (version finale)

**6 classes originales** (TrashNet + TACO) + **2 nouvelles** (Open Images) :
- ✅ cardboard, glass, metal, paper, plastic, trash
- ✅ 💡 bulb (ampoules) — données suffisantes
- ✅ 📱 electronic — données suffisantes
- ❌ battery / medicine retirés (pas assez de données)

## Avant de commencer
1. **GPU T4** : Settings → Accelerator → GPU T4 x2
2. **Internet** : Settings → Internet → On
3. **Add Input** : ajoute ton dataset `waste-ai-v4` (contenant `waste_ai_v4.pt`)

## Étape 1 — Installation

In [ ]:
%pip install timm albumentations tqdm -q

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')
if DEVICE != 'cuda':
    print('ATTENTION : Active le GPU T4 dans Settings → Accelerator')

## Étape 2 — Téléchargement TrashNet

In [ ]:
import os

TRASHNET_DIR = '/kaggle/working/trashnet'

if not os.path.exists(f'{TRASHNET_DIR}/dataset-resized'):
    print('Téléchargement TrashNet...')
    !wget -q https://github.com/garythung/trashnet/raw/master/data/dataset-resized.zip -O /kaggle/working/dataset-resized.zip
    !unzip -q /kaggle/working/dataset-resized.zip -d {TRASHNET_DIR}/
    print('OK')
else:
    print('TrashNet déjà présent')

for c in sorted([c for c in os.listdir(f'{TRASHNET_DIR}/dataset-resized') if not c.startswith('.')]):
    print(f'  {c}: {len(os.listdir(f"{TRASHNET_DIR}/dataset-resized/{c}"))} images')

## Étape 3 — Téléchargement TACO

In [ ]:
import os, json, requests
from tqdm import tqdm

TACO_DIR = '/kaggle/working/TACO'
os.makedirs(f'{TACO_DIR}/data/images', exist_ok=True)

ann_path = f'{TACO_DIR}/data/annotations.json'
if not os.path.exists(ann_path):
    r = requests.get('https://raw.githubusercontent.com/pedropro/TACO/master/data/annotations.json', timeout=30)
    with open(ann_path, 'w') as f:
        f.write(r.text)

with open(ann_path) as f:
    taco_data = json.load(f)

downloaded, errors = 0, 0
for img in tqdm(taco_data['images'], desc='TACO'):
    out = f"{TACO_DIR}/data/images/{img['id']}.jpg"
    if os.path.exists(out):
        continue
    url = img.get('flickr_url') or img.get('coco_url', '')
    if not url:
        continue
    try:
        r = requests.get(url, timeout=15)
        if r.status_code == 200:
            with open(out, 'wb') as f:
                f.write(r.content)
            downloaded += 1
    except:
        errors += 1

total = len([f for f in os.listdir(f'{TACO_DIR}/data/images') if f.endswith('.jpg')])
print(f'TACO : {downloaded} téléchargées | {errors} erreurs | {total} total')

## Étape 4 — Téléchargement Open Images (bulb + electronic uniquement)

In [ ]:
import pandas as pd
import os

OI_DIR = '/kaggle/working/openimages'
os.makedirs(f'{OI_DIR}/bulb', exist_ok=True)
os.makedirs(f'{OI_DIR}/electronic', exist_ok=True)

# Seulement les 2 classes fiables
TARGET_CLASSES = {
    'Light bulb':     'bulb',
    'Mobile phone':   'electronic',
    'Laptop':         'electronic',
    'Television':     'electronic',
    'Tablet computer':'electronic',
}

MAX_PER_CLASS = 400

print('Chargement métadonnées Open Images...')
desc = pd.read_csv(
    'https://storage.googleapis.com/openimages/v6/oidv6-class-descriptions.csv',
    header=None, names=['id', 'name']
)
name_to_id  = {row['name']: row['id'] for _, row in desc.iterrows() if row['name'] in TARGET_CLASSES}
id_to_class = {v: TARGET_CLASSES[k] for k, v in name_to_id.items()}
print(f'Classes : {name_to_id}')

labels_val = pd.read_csv(
    'https://storage.googleapis.com/openimages/v5/validation-annotations-human-imagelabels.csv'
)
pos = labels_val[
    labels_val['LabelName'].isin(id_to_class.keys()) &
    (labels_val['Confidence'] == 1)
]
print(f'Annotations validation : {len(pos)}')

In [ ]:
import boto3
from botocore import UNSIGNED
from botocore.config import Config
from tqdm import tqdm

s3     = boto3.client('s3', config=Config(signature_version=UNSIGNED), region_name='us-east-1')
BUCKET = 'open-images-dataset'
counts = {'bulb': 0, 'electronic': 0}

# --- Validation set ---
for label_id, cls in id_to_class.items():
    if counts[cls] >= MAX_PER_CLASS:
        continue
    img_ids = pos[pos['LabelName'] == label_id]['ImageID'].unique()
    print(f'Validation {cls} ({label_id}) : {len(img_ids)} images')
    for img_id in tqdm(img_ids, desc=cls):
        if counts[cls] >= MAX_PER_CLASS:
            break
        out = f'{OI_DIR}/{cls}/{img_id}.jpg'
        if os.path.exists(out):
            counts[cls] += 1
            continue
        try:
            s3.download_file(BUCKET, f'validation/{img_id}.jpg', out)
            counts[cls] += 1
        except:
            pass

# --- Train set si pas assez ---
classes_insuffisantes = [c for c in counts if counts[c] < 100]
if classes_insuffisantes:
    print(f'Complément depuis train set pour : {classes_insuffisantes}')
    labels_train = pd.read_csv(
        'https://storage.googleapis.com/openimages/v5/train-annotations-human-imagelabels.csv',
        nrows=3_000_000
    )
    pos_train = labels_train[
        labels_train['LabelName'].isin(id_to_class.keys()) &
        (labels_train['Confidence'] == 1)
    ]
    for label_id, cls in id_to_class.items():
        if cls not in classes_insuffisantes or counts[cls] >= MAX_PER_CLASS:
            continue
        img_ids = pos_train[pos_train['LabelName'] == label_id]['ImageID'].unique()
        print(f'Train {cls} : {len(img_ids)} images dispo')
        for img_id in tqdm(img_ids, desc=f'train-{cls}'):
            if counts[cls] >= MAX_PER_CLASS:
                break
            out = f'{OI_DIR}/{cls}/{img_id}.jpg'
            if os.path.exists(out):
                counts[cls] += 1
                continue
            try:
                s3.download_file(BUCKET, f'train/{img_id}.jpg', out)
                counts[cls] += 1
            except:
                pass

print('\nRésultat Open Images :')
for cls in ['bulb', 'electronic']:
    print(f'  {cls}: {len(os.listdir(f"{OI_DIR}/{cls}"))} images')

## Étape 5 — Organisation (8 classes)

In [ ]:
import os, json, shutil

TACO_DIR     = '/kaggle/working/TACO'
TRASHNET_DIR = '/kaggle/working/trashnet'
OI_DIR       = '/kaggle/working/openimages'
ALL_DATA     = '/kaggle/working/all_data'

# 8 classes — ordre ALPHABÉTIQUE (ImageFolder)
# bulb=0, cardboard=1, electronic=2, glass=3, metal=4, paper=5, plastic=6, trash=7
CLASS_NAMES = ['bulb', 'cardboard', 'electronic', 'glass', 'metal', 'paper', 'plastic', 'trash']

for cls in CLASS_NAMES:
    os.makedirs(f'{ALL_DATA}/{cls}', exist_ok=True)

# TrashNet → 6 classes
TRASHNET_MAP = {'cardboard': 'cardboard', 'glass': 'glass', 'metal': 'metal',
                'paper': 'paper', 'plastic': 'plastic', 'trash': 'trash'}
for src_cls, dst_cls in TRASHNET_MAP.items():
    src_dir = f'{TRASHNET_DIR}/dataset-resized/{src_cls}'
    if not os.path.exists(src_dir):
        continue
    for fname in os.listdir(src_dir):
        if fname.startswith('.'):
            continue
        dst = f'{ALL_DATA}/{dst_cls}/trashnet_{fname}'
        if not os.path.exists(dst):
            shutil.copy(f'{src_dir}/{fname}', dst)

# TACO → 6 classes
TACO_TO_CLASS = {
    'Plastic bottle': 'plastic', 'Plastic bag & wrapper': 'plastic',
    'Plastic container': 'plastic', 'Plastic cup': 'plastic',
    'Plastic lid': 'plastic', 'Styrofoam piece': 'plastic',
    'Paper': 'paper', 'Paper bag': 'paper', 'Newspaper': 'paper',
    'Cardboard': 'cardboard', 'Carton': 'cardboard', 'Drink carton': 'cardboard',
    'Glass bottle': 'glass', 'Broken glass': 'glass', 'Glass jar': 'glass',
    'Aluminium foil': 'metal', 'Metal bottle cap': 'metal',
    'Drink can': 'metal', 'Food can': 'metal',
    'Cigarette': 'trash', 'Rope & strings': 'trash', 'Unlabeled litter': 'trash',
}
with open(f'{TACO_DIR}/data/annotations.json') as f:
    taco_data = json.load(f)
cat_id_to_name = {c['id']: c['name'] for c in taco_data['categories']}
for ann in taco_data['annotations']:
    target = TACO_TO_CLASS.get(cat_id_to_name.get(ann['category_id'], ''))
    if not target:
        continue
    src = f"{TACO_DIR}/data/images/{ann['image_id']}.jpg"
    dst = f"{ALL_DATA}/{target}/taco_{ann['image_id']}_{ann['id']}.jpg"
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy(src, dst)

# Open Images → bulb + electronic
for cls in ['bulb', 'electronic']:
    src_dir = f'{OI_DIR}/{cls}'
    if not os.path.exists(src_dir):
        continue
    for fname in os.listdir(src_dir):
        dst = f'{ALL_DATA}/{cls}/oi_{fname}'
        if not os.path.exists(dst):
            shutil.copy(f'{src_dir}/{fname}', dst)

print('Dataset 8 classes :')
total = 0
for cls in CLASS_NAMES:
    n = len(os.listdir(f'{ALL_DATA}/{cls}'))
    total += n
    print(f'  {cls:12s}: {n:4d} images')
print(f'  TOTAL       : {total}')

## Étape 6 — Dataset & Augmentation

In [ ]:
import numpy as np
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader, random_split, WeightedRandomSampler
from torchvision import datasets
import albumentations as A
from albumentations.pytorch import ToTensorV2

ALL_DATA    = '/kaggle/working/all_data'
CLASS_NAMES = ['bulb', 'cardboard', 'electronic', 'glass', 'metal', 'paper', 'plastic', 'trash']
NUM_CLASSES = len(CLASS_NAMES)
DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'

train_transform = A.Compose([
    A.RandomResizedCrop(size=(260, 260), scale=(0.4, 1.0)),
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=45, p=0.5),
    A.OneOf([
        A.GaussNoise(std_range=(0.05, 0.2)),
        A.GaussianBlur(blur_limit=5),
        A.MotionBlur(blur_limit=5),
    ], p=0.4),
    A.OneOf([
        A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3),
        A.HueSaturationValue(hue_shift_limit=20, sat_shift_limit=30),
        A.CLAHE(clip_limit=4),
    ], p=0.5),
    A.CoarseDropout(num_holes_range=(1, 6), hole_height_range=(8, 32), hole_width_range=(8, 32), p=0.3),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(height=260, width=260),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

class WasteDataset(Dataset):
    def __init__(self, root, transform=None):
        self.data    = datasets.ImageFolder(root)
        self.transform = transform
        self.classes = self.data.classes
        self.targets = [s[1] for s in self.data.samples]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        path, label = self.data.samples[idx]
        try:
            image = np.array(Image.open(path).convert('RGB'))
        except Exception:
            image = np.zeros((260, 260, 3), dtype=np.uint8)
        if self.transform:
            image = self.transform(image=image)['image']
        return image, label

full_ds = WasteDataset(ALL_DATA, transform=train_transform)
print(f'Classes : {full_ds.classes}')
assert full_ds.classes == CLASS_NAMES, f'Ordre inattendu : {full_ds.classes}'

n_train = int(0.8 * len(full_ds))
n_val   = len(full_ds) - n_train
train_set, val_set = random_split(full_ds, [n_train, n_val],
                                   generator=torch.Generator().manual_seed(42))

# Sampler équilibré
class_counts      = [len(os.listdir(f'{ALL_DATA}/{c}')) for c in CLASS_NAMES]
max_count         = max(class_counts)
class_w_sample    = [max_count / max(c, 1) for c in class_counts]
train_targets     = [full_ds.targets[i] for i in train_set.indices]
sample_weights    = [class_w_sample[t] for t in train_targets]
sampler           = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

train_loader = DataLoader(train_set, batch_size=24, sampler=sampler,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_set,   batch_size=24, shuffle=False,    num_workers=2, pin_memory=True)

print(f'Total : {len(full_ds)} | Train : {n_train} | Val : {n_val}')
for cls, n in zip(CLASS_NAMES, class_counts):
    print(f'  {cls:12s}: {n:4d} images')

## Étape 7 — Chargement v4 + nouvelle tête 8 classes

In [ ]:
import torch
import torch.nn as nn
import timm
import os

DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'
CLASS_NAMES = ['bulb', 'cardboard', 'electronic', 'glass', 'metal', 'paper', 'plastic', 'trash']
NUM_CLASSES = len(CLASS_NAMES)

# Cherche v4 (ou v3 en fallback)
PREV_PATH = None
for root, dirs, files in os.walk('/kaggle/input'):
    for fname in ['waste_ai_v4.pt', 'waste_ai_v3.pt']:
        candidate = os.path.join(root, fname)
        if os.path.exists(candidate):
            if PREV_PATH is None or 'v4' in fname:
                PREV_PATH = candidate

if PREV_PATH is None:
    raise FileNotFoundError('Ajoute ton dataset waste-ai-v4 en input.')
print(f'Source : {PREV_PATH}')

# Charge backbone avec 6 classes (anciens poids)
model = timm.create_model('efficientnet_b2', pretrained=False, num_classes=6)
model.load_state_dict(torch.load(PREV_PATH, map_location='cpu'))
print('Backbone v4 chargé (6 classes)')

# Remplace la tête par 8 classes
in_features      = model.classifier.in_features
model.classifier = nn.Linear(in_features, NUM_CLASSES)
model            = model.to(DEVICE)
print(f'Nouvelle tête : {in_features} → {NUM_CLASSES} classes')

## Étape 8 — Entraînement

In [ ]:
import torch.nn as nn
import os

DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'
CLASS_NAMES = ['bulb', 'cardboard', 'electronic', 'glass', 'metal', 'paper', 'plastic', 'trash']

os.makedirs('/kaggle/working/checkpoints', exist_ok=True)

class_counts = [len(os.listdir(f'/kaggle/working/all_data/{c}')) for c in CLASS_NAMES]
total        = sum(class_counts)
loss_weights = torch.tensor(
    [total / (len(CLASS_NAMES) * max(n, 1)) for n in class_counts]
).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=loss_weights, label_smoothing=0.1)

print('Poids de loss :')
for cls, w in zip(CLASS_NAMES, loss_weights.cpu()):
    print(f'  {cls:12s}: {w:.2f}')

history  = {'train_loss': [], 'val_acc': []}
best_acc = 0.0


def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, correct, total = 0, 0, 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            if train:
                optimizer.zero_grad()
            out  = model(imgs)
            loss = criterion(out, labels)
            if train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item()
            correct    += (out.argmax(1) == labels).sum().item()
            total      += labels.size(0)
    return total_loss / len(loader), correct / total * 100


# Phase 1 — tête seulement (5 epochs)
print('\nPhase 1 : tête (5 epochs)')
for name, param in model.named_parameters():
    param.requires_grad = ('classifier' in name)

optimizer = torch.optim.Adam(model.classifier.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=5)

for epoch in range(5):
    loss, _ = run_epoch(train_loader, train=True)
    _, acc  = run_epoch(val_loader,   train=False)
    scheduler.step()
    history['train_loss'].append(loss)
    history['val_acc'].append(acc)
    print(f'  Epoch {epoch+1}/5 | Loss: {loss:.4f} | Val Acc: {acc:.1f}%')

# Phase 2 — fine-tuning complet (15 epochs)
print('\nPhase 2 : fine-tuning complet (15 epochs)')
for param in model.parameters():
    param.requires_grad = True

optimizer = torch.optim.AdamW([
    {'params': model.blocks.parameters(),     'lr': 3e-5},
    {'params': model.conv_stem.parameters(),  'lr': 3e-5},
    {'params': model.classifier.parameters(), 'lr': 1e-4},
], weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=15)

for epoch in range(15):
    loss, _ = run_epoch(train_loader, train=True)
    _, acc  = run_epoch(val_loader,   train=False)
    scheduler.step()
    history['train_loss'].append(loss)
    history['val_acc'].append(acc)
    tag = ''
    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), '/kaggle/working/checkpoints/waste_ai_v6.pt')
        tag = ' ✓ sauvegardé'
    print(f'  Epoch {epoch+1:2d}/15 | Loss: {loss:.4f} | Val Acc: {acc:.1f}%{tag}')

print(f'\nMeilleure accuracy : {best_acc:.1f}%')

## Étape 9 — Évaluation

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

CLASS_NAMES = ['bulb', 'cardboard', 'electronic', 'glass', 'metal', 'paper', 'plastic', 'trash']
DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'

model.load_state_dict(torch.load('/kaggle/working/checkpoints/waste_ai_v6.pt', map_location=DEVICE))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in val_loader:
        preds = model(imgs.to(DEVICE)).argmax(1).cpu()
        all_preds.extend(preds.numpy())
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))

cm = confusion_matrix(all_labels, all_preds)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].plot(history['train_loss'], color='#e74c3c', label='Loss')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].grid(alpha=0.3)
ax2 = axes[0].twinx()
ax2.plot(history['val_acc'], color='#2ecc71', linestyle='--', label='Val Acc')
ax2.set_ylabel('Accuracy (%)')

sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[1])
axes[1].set_title('Matrice de confusion — Waste AI v6 (8 classes)')
axes[1].set_ylabel('Réel')
axes[1].set_xlabel('Prédit')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('/kaggle/working/checkpoints/eval_v6.png', dpi=150)
plt.show()

## Étape 10 — Export

In [ ]:
import shutil
shutil.copy('/kaggle/working/checkpoints/waste_ai_v6.pt', '/kaggle/working/waste_ai_v6.pt')
print('waste_ai_v6.pt copié à la racine — télécharge depuis le panneau Output')
print('Place dans : waste-ai/model/checkpoints/waste_ai_v6.pt')